In [1]:
import pandas as pd
import numpy as np
from causalexplain import GraphDiscovery
from graphviz import Digraph
from pathlib import Path
import re

def draw_graphviz_dag(adj, nodes,labels, out_path, engine="dot"):
    #guards to check for adjacency matrix dimension
    adj = np.asarray(adj)
    print("draw_graphviz_dag adj ndim:", adj.ndim, "shape:", adj.shape)

    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]
    nodes = list(nodes)[:n]
    
    # Restore original labels where possible
    labels = labels

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    g.attr(rankdir="TB")  # left-to-right; change to "TB" if you prefer top-down
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10"
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7"
    )

    # Add nodes with restored labels
    for clean_name, label in zip(nodes, labels):
        g.node(clean_name, label=label)

    # Add edges
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)



#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_ReX"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
# Map from cleaned -> original for restoring labels in plots
def clean_and_encode_df(df: pd.DataFrame):
    """
    - Drop rows with NA.
    - Clean column names for algorithms (letters+digits, start with letter).
    - One-hot encode non-numeric columns.
    Returns:
      df_enc: encoded numeric DataFrame
      clean_to_orig: dict {clean_name: original_name}
    """
    df = df.dropna().copy()
    if df.empty:
        raise ValueError("Data frame is empty after dropna().")

    # 1) Clean base column names
    orig_cols = list(df.columns)
    clean_cols = []
    for c in orig_cols:
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)  # remove underscores, spaces, etc.
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        clean_cols.append(c2)
    df.columns = clean_cols
    clean_to_orig = dict(zip(clean_cols, orig_cols))

    # 2) One-hot encode non-numeric columns
    non_numeric = df.select_dtypes(exclude=["number"]).columns
    if len(non_numeric) > 0:
        df_enc = pd.get_dummies(df, columns=list(non_numeric), drop_first=False, dtype=float)
    else:
        df_enc = df.astype(float)

    return df_enc, clean_to_orig

In [3]:
def dot_to_adjacency(dot_path):
    G = nx.drawing.nx_pydot.read_dot(str(dot_path))
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    p = len(nodes)
    adj = np.zeros((p, p), dtype=int)
    for u, v in G.edges():
        i = idx[u]
        j = idx[v]
        adj[i, j] = 1

    adj = np.asarray(adj)
    if adj.ndim == 1:              # safety
        adj = adj.reshape(1, 1)
    return adj, nodes

In [4]:
import networkx as nx
def run_rex(df, experiment_name="rex_exp"):
    #Graph Discovery expects a file path for dataframe
    tmp_csv = "tmp_rex_input.csv"
    df = clean_name(df)
    df.to_csv(tmp_csv, index=False) #write df to a csv temporarily

    #init graph discovery instance
    gd = GraphDiscovery(experiment_name=experiment_name, model_type="rex",csv_filename= tmp_csv)

    gd.run(quiet=True)

    dot_path = output_dir / f"{experiment_name}.dot"
    gd.export_dag(str(dot_path))

    # Read DOT with networkx and build adjacency
    adj, nodes = dot_to_adjacency(dot_path)
    print("ReX adj ndim:", adj.ndim, "shape:", adj.shape)

    return adj, nodes

In [5]:
def clean_name(df):
    #makes column names ReX friendly i.e start with letter, contain only letters and numbers
    new_cols = []
    for c in df.columns:
        # remove underscores and other non-alphanumeric
        c2 = re.sub(r'[^A-Za-z0-9]', '', c)
        # if it doesn't start with a letter, prefix with 'X'
        if not c2 or not c2[0].isalpha():
            c2 = "X" + c2
        new_cols.append(c2)
    df2 = df.copy()
    df2.columns = new_cols
    return df2

In [ ]:
csv_files = sorted(cp_root.rglob("*.csv"))

for csv_path in csv_files:
    rel = csv_path.relative_to(cp_root)
    print(f"Processing (ReX): {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue

        orig_labels = df.columns

        adj, nodes = run_rex(df, experiment_name=f"rex__{csv_path.stem}")
        print("  ReX returned adj shape:", getattr(adj, "shape", None),
              "nodes:", len(nodes))
        
        if adj is None or np.size(adj) == 0:
            print("  No adjacency matrix from ReX; skipping plot.")
            continue
            
        parts = rel.parts
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__ReX.png"
        out_path = output_dir / out_name

        draw_graphviz_dag(adj, nodes, orig_labels, out_path)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Processing (ReX): berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
  ERROR on berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv: too many indices for array: array is 1-dimensional, but 2 were indexed
Processing (ReX): berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
  ERROR on berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv: too many indices for array: array is 1-dimensional, but 2 were indexed
Processing (ReX): berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv
Setting tolerance to 0.3, as no true_graph was provided.
Setting tolerance to 0.3, as no true_graph was provided.
  ERROR on berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv: too many indices for